# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [15]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one: item.

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [16]:
import pandas as pd

lines_df=pd.json_normalize(
    orders_json,
    record_path='lines',
    meta=['order', 'vendor_id']
)

assert len(lines_df)==4
assert lines_df['qty'].sum()==7
lines_df

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-07


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [17]:
vendors_df=pd.DataFrame(vendors_json)

merged_df=lines_df.merge(
    vendors_df,
    on='vendor_id',
    how='left',
    indicator=True
)
not_matched=merged_df[merged_df['_merge']=='left_only']

print("Unmatched Vendor:", not_matched['vendor_id'].unique())
print("Units Attached to Vendor:", not_matched['qty'].sum())

Unmatched Vendor: ['V-07']
Units Attached to Vendor: 3


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [18]:
merged_df['name']=merged_df['name'].fillna('Unnamed Vendor')
vendor_totals=merged_df.groupby('name')['qty'].sum()
print(vendor_totals)

name
Cav Merch         1
Hoos Burgers      3
Unnamed Vendor    3
Name: qty, dtype: int64


In [19]:
#I gave the unmatched vendor a name ("Unnamed Vendor") so that it will appear in the table and isn't excluded because of its missing name.

### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [20]:
zone_totals=merged_df.groupby('zone')['qty'].sum()

print(zone_totals)
print("Total units by zone:", zone_totals.sum())
print("Missing units:", merged_df['qty'].sum()-zone_totals.sum())

merged_df[merged_df['zone'].isna()]

zone
A    4
Name: qty, dtype: int64
Total units by zone: 4
Missing units: 3


,item,qty,order,vendor_id,name,zone,_merge
3,Hot Dog,3,3,V-07,Unnamed Vendor,NaN,left_only


In [21]:
#The zone total is 4 rather than 7 because V-07 has no zone, so its 3 units are not included in the total.

### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [22]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]


for order in ragged_json:
  order.setdefault('lines', [])

ragged_df=pd.json_normalize(
    ragged_json,
    record_path='lines',
    meta=['order', 'vendor_id']
)

print(ragged_df)

    item  qty order vendor_id
0   Soda  2.0     4      V-01
1  Fries  NaN     6      V-01


### Q6 — Validate

In [23]:
assert len(lines_df) == 4
assert lines_df['qty'].sum() == 7
assert len(merged_df) == len(lines_df), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a) One row per line item is the useful shape because certain orders, such as order 1, have multiple items. This structure makes it easier to analyze sales by item, such as determining which item sold the most units. If this weren't the case, item-level analysis would require extracting individual items from the lists.

b) A missing quantity means that the data does not exist, while a quantity of zero means the data *does* exist with that value. In Q5, for example, the quantity of fries of sold is missing, therefore it could be anything. Inserting zero for this missing value is not the right move because the real quantity could be far from zero. Thus, treating missing quantities as zero could lead to inaccuracies in item totals, potentially underestimating the number of units sold.